# Model Team A – Outputs for Cross-Team Comparison
Runs the full Team A pipeline and saves `model_comparison.csv` and `{horizon}_sparse_l1_oof.csv` for use by Team B.

In [1]:
import os, warnings
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss
warnings.filterwarnings("ignore")


In [2]:
DATA_PATH = "DS_SOURCES_PIPELINE/master_dataset_transformed_after_1986.csv"
OUTDIR    = "model_team_a_outputs"
os.makedirs(OUTDIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df["Date"] = pd.to_datetime(df["Date"])
df = df.drop(columns=["BAA10Y_missing", "T10Y2Y_missing", "FEDFUNDS_missing", "TEDRATE"])
print(f"Loaded: {df.shape}")


Loaded: (145, 11)


In [3]:
signal_cols = ["T10Y2Y", "BAA10Y", "UNRATE_d1", "INDPRO_lgyoy", "CPIAUCSL_lgyoy", "FEDFUNDS_d1"]
edf = df.copy()
for col in signal_cols:
    for lag in [1, 2, 3]:
        edf[f"{col}_lag{lag}"] = edf[col].shift(lag)
    for win in [3, 6]:
        edf[f"{col}_roll{win}"] = edf[col].rolling(win, min_periods=win).mean()
edf["post_2008"] = (edf["Date"] >= pd.Timestamp("2008-01-01")).astype(int)
edf["post_2020"] = (edf["Date"] >= pd.Timestamp("2020-01-01")).astype(int)

target_cols   = ["Target_1Q_ahead", "Target_2Q_ahead", "Target_3Q_ahead"]
feature_cols  = [c for c in edf.columns if c not in ["Date", "USRECD"] + target_cols]
horizons      = target_cols
print(f"Rows: {len(edf)}, Features: {len(feature_cols)}")


Rows: 145, Features: 38


In [4]:
def prep_dataset(horizon):
    data = edf.dropna(subset=feature_cols + [horizon]).copy().reset_index(drop=True)
    return data, data[feature_cols], data[horizon].astype(int)

def expanding_splits(n, initial=60, test=8, step=8):
    splits, start = [], initial
    while start + test <= n:
        splits.append((np.arange(start), np.arange(start, start + test)))
        start += step
    return splits


In [5]:
# Walk-forward validation — sparse_l1 and full_l2
models = {
    "sparse_l1": LogisticRegression(penalty="l1", C=0.25, class_weight="balanced",
                                     max_iter=5000, solver="liblinear"),
    "full_l2":   LogisticRegression(penalty="l2", C=1.0,  class_weight="balanced",
                                     max_iter=5000, solver="liblinear"),
}

all_results = []
oof_store   = {}

for horizon in horizons:
    data, X, y = prep_dataset(horizon)
    splits = expanding_splits(len(data))
    for model_name, model in models.items():
        preds, ys, dates = [], [], []
        for tr, te in splits:
            imp    = SimpleImputer(strategy="median")
            scaler = StandardScaler()
            Xtr_s  = scaler.fit_transform(imp.fit_transform(X.iloc[tr]))
            Xte_s  = scaler.transform(imp.transform(X.iloc[te]))
            model.fit(Xtr_s, y.iloc[tr])
            preds.extend(model.predict_proba(Xte_s)[:, 1])
            ys.extend(y.iloc[te].tolist())
            dates.extend(data.iloc[te]["Date"].tolist())
        ys, preds = np.array(ys), np.array(preds)
        all_results.append({
            "horizon":           horizon.replace("Target_", "").replace("_ahead", ""),
            "model":             model_name,
            "oof_n":             len(ys),
            "positives":         int(ys.sum()),
            "auc":               roc_auc_score(ys, preds),
            "average_precision": average_precision_score(ys, preds),
            "brier":             brier_score_loss(ys, preds),
            "log_loss":          log_loss(ys, preds, labels=[0, 1]),
        })
        oof_store[(horizon, model_name)] = pd.DataFrame(
            {"Date": pd.to_datetime(dates), "y_true": ys, "y_prob": preds}
        ).sort_values("Date").reset_index(drop=True)
        print(f"  {horizon}  {model_name:12s}  AUC={all_results[-1]['auc']:.3f}")

print("\nDone.")


  Target_1Q_ahead  sparse_l1     AUC=0.953
  Target_1Q_ahead  full_l2       AUC=0.925
  Target_2Q_ahead  sparse_l1     AUC=0.922
  Target_2Q_ahead  full_l2       AUC=0.835
  Target_3Q_ahead  sparse_l1     AUC=0.851
  Target_3Q_ahead  full_l2       AUC=0.743

Done.


In [6]:
# Save model_comparison.csv (used by Team B comparison cell)
results_df = pd.DataFrame(all_results).sort_values(["model", "horizon"]).reset_index(drop=True)
results_df.to_csv(os.path.join(OUTDIR, "model_comparison.csv"), index=False)
print("Saved: model_comparison.csv")

# Save sparse_l1 OOF predictions per horizon (used by Team B stacking ensemble)
for horizon in horizons:
    oof_store[(horizon, "sparse_l1")].to_csv(
        os.path.join(OUTDIR, f"{horizon}_sparse_l1_oof.csv"), index=False
    )
    print(f"Saved: {horizon}_sparse_l1_oof.csv")

print("\nAll Team A outputs ready.")
display(results_df.round(4))


Saved: model_comparison.csv
Saved: Target_1Q_ahead_sparse_l1_oof.csv
Saved: Target_2Q_ahead_sparse_l1_oof.csv
Saved: Target_3Q_ahead_sparse_l1_oof.csv

All Team A outputs ready.


,horizon,model,oof_n,positives,auc,average_precision,brier,log_loss
0,1Q,full_l2,80,8,0.9253,0.4235,0.0981,1.0047
1,2Q,full_l2,80,8,0.8351,0.4443,0.1215,0.5042
2,3Q,full_l2,80,8,0.7431,0.2235,0.2005,0.8573
3,1Q,sparse_l1,80,8,0.9531,0.6282,0.0755,0.5739
4,2Q,sparse_l1,80,8,0.9219,0.5429,0.0958,0.3229
5,3Q,sparse_l1,80,8,0.8507,0.3683,0.1342,0.4062
